# 🏡 San Antonio Automated Apartment & Townhome Hunter (v5 — Clean Frontend Dashboard)
### Welcome to Your Apartment-Hunting Control Center
To keep this notebook clean and easy to customize, **all complex algorithms, KD-Tree indexers, scoring criteria, and vision models are safely tucked away in backend modules (`src/`)**. 

Use the dashboard cells below to adjust your search radius, budget, weights, and persistent taste memory!

In [ ]:
# @title 🔌 Setup: Install Dependencies & Path Resolution
import sys
import os

# Robust sys.path injection for local Jupyter / Colab execution
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)
if parent_dir not in sys.path and parent_dir != '':
    sys.path.insert(0, parent_dir)

# If running inside notebooks/ subdirectory, jump up one level to find `src/`
if not os.path.exists('src') and os.path.exists('../src'):
    os.chdir('..')
    print('📁 Changed working directory to repo root:', os.getcwd())

print('Current working directory:', os.getcwd())
print('Directory contents:', os.listdir('.'))
if 'src' in os.listdir('.'):
    print('✅ Found `src/` directory! Local module imports ready.')
else:
    print('⚠️ `src/` not found. Ensure Jupyter was launched from the `LoveWhereYouLive/` root folder.')

# Set to True ONLY when running inside Google Colab with Google Drive connected
USE_GOOGLE_DRIVE_MEMORY = False

if USE_GOOGLE_DRIVE_MEMORY:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = '/content/drive/MyDrive/LoveWhereYouLive_cache'
    PREF_FILE = '/content/drive/MyDrive/LoveWhereYouLive_preferences.json'
else:
    CACHE_DIR = 'cache'
    PREF_FILE = 'data/preferences.json'

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs('outputs', exist_ok=True)
print('✅ Frontend initialized! Local Cache Directory:', CACHE_DIR)

In [ ]:
# @title 🎛️ Customize Your Search Filters, Cutoffs, Custom Complexes & Weights
import yaml

CONFIG = {
    'workplace': {'name': 'Southwest Research Institute (SwRI)', 'coords': (29.4485, -98.6105)},
    'costco_locations': [
        {'name': 'Costco NW (5611 UTSA Blvd)', 'coords': (29.5762, -98.6001)},
        {'name': 'Costco West (191 W Loop 1604 S)', 'coords': (29.4316, -98.7115)},
        {'name': 'Costco North Central (1201 N Loop 1604 E)', 'coords': (29.6095, -98.4716)},
        {'name': 'Costco NE Selma (15330 IH-35 N)', 'coords': (29.5794, -98.3188)}
    ],
    # Optional watchlist of Zillow /apartments/ URLs or street addresses that bypass MLS.
    # Note: Any complex you inspect in the Single-Link Inspector below is ALSO automatically saved to preferences.json!
    'custom_complexes': [
        # 'https://www.zillow.com/apartments/san-antonio-tx/botanica-oak-hills/CjfHGx/',
        # 'https://www.zillow.com/apartments/san-antonio-tx/victor-at-alamo-ranch/Ck7kMg/',
        # '7716 Louis Pasteur Dr, San Antonio, TX 78229'
    ],
    'filters': {
        'location': 'San Antonio, TX',
        'min_rent': 1000,
        'max_rent': 2200,
        'min_beds': 2,
        'min_baths': 1.5,
        'past_days': 60
    },
    'cutoffs': {
        'require_in_unit_laundry_and_dishwasher': True,
        'max_swri_commute_mins': 30.0,
        'max_grocery_dist_m': 2400.0,
        'max_park_trail_dist_m': 1600.0
    },
    'weights': {
        'commute_score': 0.2,
        'grocery_score': 0.11,
        'park_trail_score': 0.11,
        'pollution_highway_score': 0.03,
        'modern_interior_score': 0.15,
        'garage_bike_score': 0.07,
        'costco_score': 0.04,
        'water_softener_score': 0.05,
        'pet_flooring_score': 0.02,
        'effective_rent_score': 0.05,
        'noise_buffer_score': 0.03,
        'recency_score': 0.06
    }
}
CONFIG['apify_zillow_file'] = os.path.join(CACHE_DIR, r'parseforge_zillow-rentals-scraper-RunID-BaBmq41vfnneuUfmf.json')
print('✅ Dashboard filters, workplace, custom complexes watchlist, and weights configured!')

In [ ]:
# @title 🚀 Run Pipeline (Vectorized Spatial Sieve + 12-Criteria + 5-Axis CLIP Vision)
from src.pipeline import run_pipeline

# Executes 3-Stage Funnel:
# Stage 1: Vectorized NumPy & KD-Tree Sieve (prunes 5,000+ listings by hard commute/grocery/park cutoffs in <5ms)
# Stage 2: Full 12-Criteria Spatial & Text Scoring on top contenders (capped at `max_scored_candidates`)
# Stage 3: 5-Axis CLIP Kitchen Vision Grader on top `clip_max_candidates` + top Apify complexes + pinned favorites
scored_df = run_pipeline(
    config_dict=CONFIG,
    cache_dir=CACHE_DIR,
    pref_file=PREF_FILE,
    run_local_clip=True,
    clip_max_candidates=35,       # Max top properties to grade with CLIP vision
    enforce_cutoffs=True,         # True = prune listings violating CONFIG['cutoffs'] in <5ms
    max_scored_candidates=300     # Cap total rows entering the full 12-criteria table
)

scored_df[['listing_status', 'composite_score', 'formatted_address', 'list_price', 'property_age_years', 'modern_interior_score', 'swri_rush_hour_mins', 'nearest_grocery']].head(15)

In [ ]:
# @title ⭐ Manage Persistent Taste Memory (Favorites, Custom Complexes & Blacklists)
from src.preference_memory import record_feedback, load_preferences

# 1. Add a favorite property to your shortlist (pins to outputs/favorites_shortlist.csv):
# record_feedback(address='123 Example St, San Antonio, TX', liked=True, reason='Great kitchen & garage', path=PREF_FILE)

# 2. Permanently blacklist an entire apartment complex name across all future searches:
# record_feedback(liked=False, blacklist_complex_name='Villas at Example', path=PREF_FILE)

prefs = load_preferences(PREF_FILE)
print('⭐ Current Favorites Shortlist:', prefs['taste_profile']['liked_property_addresses'])
print('🏢 Auto-Tracked Custom Complexes:', [c.get('formatted_address') for c in prefs['taste_profile'].get('tracked_custom_listings', [])])
print('🚫 Blacklisted Complexes:', prefs['taste_profile']['blacklisted_complex_keywords'])

In [ ]:
# @title 🔍 Single-Link / Batch Inspector & Automated Master Ledger Injector
from src.inspector import inspect_and_explain_listing
from src.criteria import ACTIVE_CRITERIA
from src.criteria.vision_grader import score_photos_with_local_clip
from src.preference_memory import load_preferences, record_feedback

# Paste a single URL/address OR a Python list of URLs/addresses below:
# Every inspected complex is AUTOMATICALLY saved to preferences.json AND ranked into san_antonio_ranked_rentals_v4.csv!
TARGET_URL_OR_ADDRESS = "" 
MANUAL_KITCHEN_PHOTO_URL = "" # Optional: Right-click listing photo -> Copy Image Address and paste here

if TARGET_URL_OR_ADDRESS:
    prefs = load_preferences(PREF_FILE)
    active_trees = TREES if 'TREES' in globals() else (TREES if 'TREES' in locals() else {})
    
    waterfall_df = inspect_and_explain_listing(
        TARGET_URL_OR_ADDRESS,
        raw_df=raw if 'raw' in globals() else (raw if 'raw' in locals() else None),
        ctx={'config': CONFIG, 'trees': active_trees},
        active_criteria=ACTIVE_CRITERIA,
        vision_fn=score_photos_with_local_clip,
        liked_urls=prefs['taste_profile'].get('liked_kitchen_photo_urls', []),
        disliked_urls=prefs['taste_profile'].get('disliked_kitchen_photo_urls', []),
        clip_cache_path=os.path.join(CACHE_DIR, 'clip_photo_cache.pkl'),
        cache_dir=CACHE_DIR,
        manual_kitchen_photo_url=MANUAL_KITCHEN_PHOTO_URL.strip() if MANUAL_KITCHEN_PHOTO_URL.strip() else None,
        auto_append_to_ledger=True,
        ledger_csv_path='outputs/san_antonio_ranked_rentals_v4.csv',
        favorites_csv_path='outputs/favorites_shortlist.csv',
        add_to_favorites=False,
        add_to_blacklist=False,
        pref_file=PREF_FILE,
        record_feedback_fn=record_feedback
    )
    display(waterfall_df)
else:
    print('💡 Paste any listing URL, Zillow /apartments/ link, or street address into `TARGET_URL_OR_ADDRESS` above and run this cell!')

## 📋 Next Steps: Review Candidates & Batch-Sync Favorites
Instead of manually recording individual listings one-by-one, use the workflow below to inspect `outputs/san_antonio_ranked_rentals_v4.csv`, filter top contenders, open `outputs/visual_kitchen_audit.html`, and batch-sync all properties you want to keep directly into persistent taste memory!

In [ ]:
# @title 🔎 Quick Filter & Contender Viewer (from Ranked Master CSV)
import pandas as pd
import os

ranked_csv_path = 'outputs/san_antonio_ranked_rentals_v4.csv'

if os.path.exists(ranked_csv_path):
    df = pd.read_csv(ranked_csv_path)
    display_cols = [
        'listing_status', 'composite_score', 'formatted_address', 'list_price',
        'beds', 'full_baths', 'swri_rush_hour_mins', 'nearest_grocery',
        'grocery_bike_mins', 'modern_interior_score', 'garage_bike_features', 'property_url'
    ]
    valid_cols = [c for c in display_cols if c in df.columns]
    
    # Display Top 15 ranked candidates
    print(f'📊 Loaded {len(df)} ranked properties from {ranked_csv_path}')
    display(df[valid_cols].head(15))
    
    # Check if a visual audit HTML was generated
    audit_html = 'outputs/visual_kitchen_audit.html'
    if os.path.exists(audit_html):
        print(f'📷 Visual Kitchen Audit card grid available at: {os.path.abspath(audit_html)}')
else:
    print(f'⚠️ File not found: {ranked_csv_path}. Run Step 3 first to generate the master ranked list!')

In [ ]:
# @title ⭐ Batch-Sync Kept Properties from CSV into Memory & Shortlist
# Tip: Open outputs/san_antonio_ranked_rentals_v4.csv in Excel / Sheets,
# add a column named 'keep' (or 'status') and put 'yes' / 'fav' / '1' on listings you want to tour.
# Then run this cell to sync them all automatically!
import os
import pandas as pd
from src.preference_memory import record_feedback

ranked_csv_path = 'outputs/san_antonio_ranked_rentals_v4.csv'
fav_out_path = 'outputs/favorites_shortlist.csv'

if os.path.exists(ranked_csv_path):
    df = pd.read_csv(ranked_csv_path)
    
    # Check for a 'keep' or 'status' column
    target_col = None
    for c in ['keep', 'status', 'shortlist', 'favorite', 'fav']:
        if c in df.columns:
            target_col = c
            break
            
    if target_col is not None:
        kept_df = df[df[target_col].astype(str).str.strip().str.lower().isin(['yes', 'y', '1', 'true', 'keep', 'fav'])].copy()
        if len(kept_df) > 0:
            print(f'🔄 Syncing {len(kept_df)} selected properties into persistent preference memory ({PREF_FILE})...')
            for _, row in kept_df.iterrows():
                addr = row['formatted_address']
                note = row.get('notes', 'Added via batch CSV review')
                record_feedback(address=addr, liked=True, reason=str(note), path=PREF_FILE)
            
            # Export directly to favorites_shortlist.csv
            kept_df.to_csv(fav_out_path, index=False)
            print(f'✅ Saved {len(kept_df)} pinned favorites to {fav_out_path}!')
        else:
            print(f'ℹ️ Found column `{target_col}`, but no rows were marked with "yes", "fav", or "1".')
    else:
        print(f'💡 To use batch-sync: Open `{ranked_csv_path}`, add a column named `keep`, mark desired rows with `yes`, and re-run this cell!')
else:
    print(f'⚠️ File `{ranked_csv_path}` does not exist yet. Run Step 3 first!')